In [ ]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

In [ ]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [2]:
import json
import os
from datasets import Dataset

# Define RunPod workspace paths
WORKSPACE_DIR = "/workspace/"
MICRO_BATCH_PATH = os.path.join(WORKSPACE_DIR, "qwen25_vl_full_sft_optimized.json") # Update with your exact filename
LOCAL_IMAGE_DIR = os.path.join(WORKSPACE_DIR, "balanced_images_512")

with open(MICRO_BATCH_PATH, "r", encoding="utf-8") as f:
    micro_batch = json.load(f)

processed_data = []
for item in micro_batch:
    # Fix the Windows backslash to Linux forward slash
    raw_img_path = item["images"][0].replace("\\", "/")
    img_filename = os.path.basename(raw_img_path)
    
    # Create the absolute Linux path
    local_img_path = os.path.join(LOCAL_IMAGE_DIR, img_filename)
    
    processed_data.append({
        "id": item["id"],
        "messages": item["messages"],
        "images": [local_img_path]
    })

hf_dataset = Dataset.from_list(processed_data)
print(f"Loaded {len(hf_dataset)} samples successfully. First image path: {hf_dataset[0]['images'][0]}")

Loaded 2456 samples successfully. First image path: /workspace/balanced_images_512/149134.png


In [6]:
import torch
from qwen_vl_utils import process_vision_info

class QwenDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        batch_messages = []
        
        for example in examples:
            raw_messages = example["messages"]
            img_path = example["images"][0] 
            
            formatted_messages = []
            image_added = False # Track if we've already added the image
            
            for msg in raw_messages:
                if msg["role"] == "user":
                    clean_text = msg["content"].replace("<|image_pad|>", "").strip()
                    
                    # Only append the image to the very first user prompt
                    if not image_added:
                        formatted_messages.append({
                            "role": "user",
                            "content": [
                                {"type": "image", "image": img_path},
                                {"type": "text", "text": clean_text}
                            ]
                        })
                        image_added = True
                    else:
                        formatted_messages.append({
                            "role": "user",
                            "content": clean_text
                        })
                else:
                    formatted_messages.append(msg)
            
            batch_messages.append(formatted_messages)

        # Apply chat template
        texts = [
            self.processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) 
            for msg in batch_messages
        ]
        
        image_inputs, video_inputs = process_vision_info(batch_messages)

        batch = self.processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )

        # ---> THE SILVER BULLET FIX <---
        # Force PyTorch autograd to track the vision tower graph during checkpointing
        if "pixel_values" in batch and batch["pixel_values"] is not None:
            batch["pixel_values"] = batch["pixel_values"].requires_grad_(True)
        if "pixel_values_videos" in batch and batch["pixel_values_videos"] is not None:
            batch["pixel_values_videos"] = batch["pixel_values_videos"].requires_grad_(True)

        # ---> SFT LABEL MASKING FIX <---
        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        im_start_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
        im_end_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
        
        for i in range(len(labels)):
            in_assistant_turn = False
            j = 0
            while j < len(labels[i]):
                token = labels[i][j]
                
                if token == im_start_id:
                    # Look ahead to see if the next token(s) decode to "assistant"
                    # We check the next 2 tokens just in case of weird tokenization splits
                    lookahead_tokens = labels[i][j+1:j+3].tolist()
                    lookahead_text = self.processor.tokenizer.decode(lookahead_tokens)
                    
                    if "assistant" in lookahead_text:
                        in_assistant_turn = True
                        labels[i][j] = -100  # Mask the <|im_start|> token itself
                        j += 1
                        continue
                
                if token == im_end_id:
                    in_assistant_turn = False
                    # We usually don't mask the <|im_end|> token so the model learns to stop
                    
                if not in_assistant_turn:
                    labels[i][j] = -100
                
                j += 1
                
        batch["labels"] = labels
        
        return batch

In [7]:
import torch
# FIX 2: Import AutoModelForVision2Seq
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint
import os

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    # FIX 1: Safely keep the vision tower in bf16 by skipping quantization
    llm_int8_skip_modules=["visual", "lm_head"]
)

# FIX 2: Let AutoModel handle the Qwen2.5 architecture mapping
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Safely freeze the vision tower without altering dtypes
for name, param in model.named_parameters():
    if "visual" in name:
        param.requires_grad = False

model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,        # alpha = 2× r is the standard stable ratio
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj"        # MLP — keep these for format learning
    ],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Initialize your custom collator (assuming it's defined in the previous cell)
data_collator = QwenDataCollator(processor)

training_args = SFTConfig(
    output_dir="/workspace/qwen_lora_3b_full_run",
    
    # Memory Management
    per_device_train_batch_size=2,    # was 1 — 3B fits larger batch
    gradient_accumulation_steps=8,    # was 16 — effective batch = 16 still 
    max_length=4096,                 
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # --- FULL RUN LEARNING PARAMETERS ---
    num_train_epochs=4,              # 4 passes over the 2,581 samples
     learning_rate=2e-4,              # <--- Lowered for fine-grained numerical accuracy
    lr_scheduler_type="cosine",      # <--- Smoothly decays the LR
    warmup_ratio=0.05,               # <--- Gentle warmup for stability
    
    save_steps=100,                  # Save less frequently to save disk space
    save_total_limit=3,
    logging_steps=10,
    
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    args=training_args,
    data_collator=data_collator,
    processing_class=processor,
)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
else:
    print("No checkpoint found. Starting training from scratch...")

trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.model.save_pretrained("/workspace/qwen_lora_3b_final")

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


No checkpoint found. Starting training from scratch...


Step,Training Loss
10,0.905354
20,0.633349
30,0.281708
40,0.179376
50,0.156320
60,0.139053
70,0.119947
80,0.117707
90,0.106962
100,0.110302


KeyboardInterrupt: 